# Comp1-传统组学

主要适配于传统组学的建模和刻画。典型的应用场景探究rad_score最最终临床诊断的作用。

数据的一般形式为(具体文件,文件夹名可以不同)：
1. `images`文件夹，存放研究对象所有的CT、MRI等数据。
2. `masks`文件夹, 存放手工（Manuelly）勾画的ROI区域。与images文件夹的文件意义对应。
3. `label.txt`文件，每个患者对应的标签，例如肿瘤的良恶性、5年存活状态等。

## Onekey步骤

1. 数据校验，检查数据格式是否正确。
2. 组学特征提取，如果第一步检查数据通过，则提取对应数据的特征。
3. 读取标注数据信息。
4. 特征与标注数据拼接。形成数据集。
5. 查看一些统计信息，检查数据时候存在异常点。
6. 正则化，将数据变化到服从 N~(0, 1)。
7. 通过相关系数，例如spearman、person等筛选出特征。
8. 构建训练集和测试集，这里使用的是随机划分，正常多中心验证，需要大家根据自己的场景构建两份数据。
9. 通过Lasso筛选特征，选取其中的非0项作为后续模型的特征。
10. 使用机器学习算法，例如LR、SVM、RF等进行任务学习。
11. 模型结果可视化，例如AUC、ROC曲线，混淆矩阵等。


In [ ]:
## 获得视频教程
import os
from onekey_algo.custom.Manager import onekey_show
onekey_show('传统组学任务')

## 一、数据校验
首先需要检查诊断数据，如果显示`检查通过！`择可以正常运行之后的，否则请根据提示调整数据。

**注意**：这里要求images和masks文件夹中的文件名必须一一对应。e.g. `1.nii.gz`为images中的一个文件，在masks文件夹必须也存在一个`1.nii.gz`文件。

当然也可以使用自定义的函数，获取解析数据。

In [ ]:
# 数据检验视频
onekey_show('传统组学任务|数据检验')

### 指定数据

此模块有3个需要自己定义的参数

1. `mydir`: 数据存放的路径。
2. `labelf`: 每个样本的标注信息文件。
3. `labels`: 要让AI系统学习的目标，例如肿瘤的良恶性、T-stage等。

In [ ]:
import os
import pandas as pd
from IPython.display import display
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
from onekey_algo import OnekeyDS as okds
from onekey_algo import get_param_in_cwd

os.makedirs('img', exist_ok=True)
os.makedirs('results', exist_ok=True)
os.makedirs('features', exist_ok=True)

# 设置任务Task前缀
task_type = 'Habitat_'
# 设置数据目录
# mydir = r'你自己数据的路径'
mydir = get_param_in_cwd('radio_dir')
if mydir == okds.ct:
    print(f'正在使用Onekey数据：{okds.ct}，如果不符合预期，请修改目录位置！')
# 对应的标签文件
group_info = get_param_in_cwd('dataset_column') or 'group'
labelf = get_param_in_cwd('label_file') or 'group.csv'
# 读取标签数据列名
labels = [get_param_in_cwd('task_column')] or ['label']

### images和masks匹配

这里要求images和masks文件夹中的文件名必须一一对应。e.g. `1.nii.gz`为images中的一个文件，在masks文件夹必须也存在一个`1.nii.gz`文件。

当然也可以使用自定义的函数，获取解析数据。

In [ ]:
from pathlib import Path
from onekey_algo.custom.components.Radiology import diagnose_3d_image_mask_settings, get_image_mask_from_dir

# 生成images和masks对，一对一的关系。也可以自定义替换。
images, masks = get_image_mask_from_dir(mydir, images=f'images', masks=get_param_in_cwd('habitat_dir'))
# diagnose_3d_image_mask_settings(images, masks, verbose=False)
print(f'获取到{len(images)}个样本。')

# 传统组学特征

使用pyradiomics提取传统组学特征，正常这块不需要修改，下面是具体的Onekey封装的接口。

```python
def extract(self, images: Union[str, List[str]], 
            masks: Union[str, List[str]], labels: Union[int, List[int]] = 1, settings=None)
"""
    * images: List结构，待提取的图像列表。
    * masks: List结构，待提取的图像对应的mask，与Images必须一一对应。
    * labels: 提取标注为什么标签的特征。默认为提取label=1的。
    * settings: 其他提取特征的参数。默认为None。

"""
```

```python
def get_label_data_frame(self, label: int = 1, column_names=None, images='images', masks='labels')
"""
    * label: 获取对应label的特征。
    * columns_names: 默认为None，使用程序设定的列名即可。
"""
```
    
```python
def get_image_mask_from_dir(root, images='images', masks='labels')
"""
    * root: 待提取特征的目录。
    * images: root目录中原始数据的文件夹名。
    * masks: root目录中标注数据的文件夹名。
"""
```


In [ ]:
# 特征提取视频
onekey_show('传统组学任务|特征提取')

In [ ]:
import warnings
import pandas as pd
import numpy as np
from onekey_algo.custom.components.Radiology import ConventionalRadiomics
from onekey_algo.custom.components.habitat import habitat_feature_fusion
 
warnings.filterwarnings("ignore")

rad_data = None
habitats = get_param_in_cwd('habitats')
rad_ = []
param_file = get_param_in_cwd('extractor_settings')
for habitat in habitats:
    if os.path.exists(f'features/rad_features_h{habitat}.csv'):
        rad_data_ = pd.read_csv(f'features/rad_features_h{habitat}.csv', header=0)
    else:
        images, masks = get_image_mask_from_dir(mydir, images=f'images', masks=get_param_in_cwd('habitat_dir'))
        radiomics = ConventionalRadiomics(param_file, correctMask=True)
        radiomics.extract(images, masks, workers=1, labels=habitat)
        rad_data_ = radiomics.get_label_data_frame(label=habitat)
        rad_data_.columns = [f"{c.replace('-', '_')}_h{habitat}" if c != 'ID' else 'ID' for c in rad_data_.columns]
        rad_data_.to_csv(f'features/rad_features_h{habitat}.csv', header=True, index=False)
    rad_.append(rad_data_)

rad_data = habitat_feature_fusion(*rad_, mode=get_param_in_cwd('fusion_mode'))
rad_data.to_csv('features/rad_features.csv', index=False)
rad_data

## 特征统计

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sorted_counts = pd.DataFrame([c.split('_')[-3] for c in rad_data.columns if c !='ID']).value_counts()
sorted_counts = pd.DataFrame(sorted_counts, columns=['count']).reset_index()
sorted_counts = sorted_counts.sort_values(0)
sorted_counts.columns = ['feature_group', 'count']
display(sorted_counts)

plt.figure(figsize=(20, 10))
ax = plt.subplot(121)
plt.pie(sorted_counts['count'], labels=[i for i in sorted_counts['feature_group']], startangle=0,
        counterclock = False, autopct = '%.1f%%')
# plt.bar_label(bar.containers[0])
ax = plt.subplot(122)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
bar = sns.barplot(data=sorted_counts, x='feature_group', y='count', )
plt.savefig(f'img/{task_type}feature_ratio.svg', bbox_inches = 'tight')

## 标注数据

数据以csv格式进行存储，这里如果是其他格式，可以使用自定义函数读取出每个样本的结果。

要求label_data为一个`DataFrame`格式，包括ID列以及后续的labels列，可以是多列，支持Multi-Task。

In [ ]:
group_info = 'group'
label_data = pd.read_csv(labelf)
label_data['ID'] = label_data['ID'].map(lambda x: f"{x}.nii.gz" if not (f"{x}".endswith('.nii.gz') or  f"{x}".endswith('.nii')) else x)
label_data = label_data[['ID', group_info] + labels]
label_data

## 特征拼接 

将标注数据`label_data`与`rad_data`进行合并，得到训练数据。

**注意：** 
1. 需要删掉ID这一列
2. 如果发现数据少了，需要自行检查数据是否匹配。

In [ ]:
# 删掉ID这一列。
from onekey_algo.custom.utils import print_join_info
print_join_info(rad_data, label_data)
combined_data = pd.merge(rad_data, label_data, on=['ID'], how='inner')
# combined_data[['ID'] + selected_features[0]].to_csv('features/sel_habitat.csv', index=False)
ids = combined_data['ID']
combined_data = combined_data.drop(['ID'], axis=1)
display(combined_data[labels].value_counts())
display(combined_data[group_info].value_counts())
combined_data

## 获取到数据的统计信息

1. count，统计样本个数。
2. mean、std, 对应特征的均值、方差
3. min, 25%, 50%, 75%, max，对应特征的最小值，25,50,75分位数，最大值。

In [ ]:
combined_data.describe()

## 正则化

`normalize_df` 为onekey中正则化的API，将数据变化到0均值1方差。正则化的方法为

$column = \frac{column - mean}{std}$

In [ ]:
from onekey_algo.custom.components.comp1 import normalize_df
data = normalize_df(combined_data, not_norm=labels, group=group_info, use_train=True, verbose=True)
data = data.dropna(axis=1)
data.describe()

## 统计检验

通过ttest或者utest进行特征筛选。

**注意** ：此步骤不是论文的标配，所以用不用在自己的选择，可以通过修改pvalue的值进行调整，默认是0.05为显著。

In [ ]:
import seaborn as sns
from onekey_algo.custom.components.stats import clinic_stats

sub_data = data[data[group_info] == 'train']
stats = clinic_stats(sub_data, stats_columns=list(data.columns[0:-2]), label_column=labels[0], 
                     continuous_columns=list(data.columns[0:-2]))
stats

#### 输出特征分布的图

In [ ]:
import matplotlib.pyplot as plt

def map2float(x):
    try:
        return float(str(x)[1:])
    except:
        return 1

stats[['pvalue']] = stats[['pvalue']].applymap(map2float)
stats[['group']] = stats[['feature_name']].applymap(lambda x: x.split('_')[-3])
stats = stats[['feature_name', 'pvalue', 'group']]
g = sns.catplot(x="group", y="pvalue", data=stats, kind="violin")
g.fig.set_size_inches(15,10)
sns.stripplot(x="group", y="pvalue", data=stats, ax=g.ax, color='black')
plt.savefig(f'img/{task_type}Rad_feature_stats.svg', bbox_inches = 'tight')

#### 调整pvalue进行筛选。

In [ ]:
pvalue = 0.05
sel_feature = list(stats[stats['pvalue'] < pvalue]['feature_name']) + labels + [group_info]
sub_data = sub_data[sel_feature]
sub_data

### 相关系数

计算相关系数的方法有3种可供选择
1. pearson （皮尔逊相关系数）: standard correlation coefficient

2. kendall (肯德尔相关性系数) : Kendall Tau correlation coefficient

3. spearman (斯皮尔曼相关性系数): Spearman rank correlation

三种相关系数参考：https://blog.csdn.net/zmqsdu9001/article/details/82840332

In [ ]:
# 相关系数视频
onekey_show('传统组学任务|相关系数')

In [ ]:
pearson_corr = sub_data[[c for c in sub_data.columns if c not in labels]].corr('pearson')
# kendall_corr = data[[c for c in data.columns if c not in labels]].corr('kendall')
# spearman_corr = data[[c for c in data.columns if c not in labels]].corr('spearman')

### 相关系数可视化

通过修改变量名，可以可视化不同相关系数下的相关矩阵。

**注意**：当特征特别多的时候（大于100），尽量不要可视化，否则运行时间会特别长。

In [ ]:
import seaborn as sns
from onekey_algo.custom.components.comp1 import draw_matrix

if data.shape[1] < 100:
    plt.figure(figsize=(50.0, 40.0))

    # 选择可视化的相关系数
    draw_matrix(pearson_corr, annot=True, cmap='YlGnBu', cbar=False)
    plt.savefig(f'img/{task_type}Rad_feature_corr.svg', bbox_inches = 'tight')

### 聚类分析

通过修改变量名，可以可视化不同相关系数下的相聚类分析矩阵。

注意：当特征特别多的时候（大于100），尽量不要可视化，否则运行时间会特别长。

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

if data.shape[1] < 100:
    pp = sns.clustermap(pearson_corr, linewidths=.5, figsize=(50.0, 40.0), cmap='YlGnBu')
    plt.setp(pp.ax_heatmap.get_yticklabels(), rotation=0)
    plt.savefig(f'img/{task_type}Rad_feature_cluster.svg', bbox_inches = 'tight')

### 特征筛选 -- 相关系数

根据相关系数，对于相关性比较高的特征（一般文献取corr>0.9），两者保留其一。

```python
def select_feature(corr, threshold: float = 0.9, keep: int = 1, topn=10, verbose=False):
    """
    * corr, 相关系数矩阵。
    * threshold，筛选的相关系数的阈值，大于阈值的两者保留其一（可以根据keep修改，可以是其二...）。默认阈值为0.9
    * keep，可以选择大于相关系数，保留几个，默认只保留一个。
    * topn, 每次去掉多少重复特征。
    * verbose，是否打印日志
    """
```

In [ ]:
# 特征筛选视频
onekey_show('传统组学任务|特征筛选')

In [ ]:
import math
import numpy as np
import pandas as pd
from scipy.stats import pointbiserialr
from onekey_algo.custom.components.comp1 import select_feature_mrmr
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold

train_data = data[data[group_info] == 'train'].copy()
test_data = data[data[group_info] != 'train'].copy()

X_train_raw = train_data.drop(labels + [group_info], axis=1)
y_train_raw = train_data[labels[0]]

def select_feature_by_corr_with_label(corr_matrix, X_data, y_data, threshold=0.9):
    corr_matrix = corr_matrix.abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop = set()
    feature_corr_with_label = {}
    for col in X_data.columns:
        try:
            corr, _ = pointbiserialr(X_data[col], y_data)
            feature_corr_with_label[col] = abs(corr)
        except:
            feature_corr_with_label[col] = 0
    for col in upper.columns:
        for row in upper.index:
            if upper.loc[row, col] > threshold:
                if feature_corr_with_label[row] >= feature_corr_with_label[col]:
                    to_drop.add(col)
                else:
                    to_drop.add(row)
    return [col for col in X_data.columns if col not in to_drop]

pearson_corr_train = X_train_raw.corr('pearson')
sel_feat_pearson = select_feature_by_corr_with_label(pearson_corr_train, X_train_raw, y_train_raw, threshold=0.9)

mrmr_sel_feature_num = 10
sel_feat_mrmr = select_feature_mrmr(train_data[sel_feat_pearson + labels], num_features=mrmr_sel_feature_num)

X_train_lr = train_data[sel_feat_mrmr].values
y_train_lr = y_train_raw.values.ravel()

C_range = np.logspace(-3, 1, 50)
mean_auc_scores = []
coef_paths = []
cv_outer = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for C in C_range:
    lr = LogisticRegression(penalty='l1', solver='saga', C=C, max_iter=10000, random_state=42)
    auc_scores = cross_val_score(lr, X_train_lr, y_train_lr, cv=cv_outer, scoring='roc_auc')
    mean_auc_scores.append(np.mean(auc_scores))
    lr.fit(X_train_lr, y_train_lr)
    coef_paths.append(lr.coef_.flatten())

best_idx = np.argmax(mean_auc_scores)
best_C = C_range[best_idx]

final_lr = LogisticRegression(penalty='l1', solver='saga', C=best_C, max_iter=10000, random_state=42)
final_lr.fit(X_train_lr, y_train_lr)
coef = final_lr.coef_.flatten()
selected_features_lr = [sel_feat_mrmr[i] for i, c in enumerate(coef) if abs(c) > 0]

sel_feature = selected_features_lr + labels + [group_info]
sel_data = data[sel_feature]

### 过滤特征

通过`sel_feature`过滤出筛选出来的特征。

In [ ]:
sel_data = data[sel_feature]
sel_data

### 样本可视化

根据特征和label信息，将rad features降维到2维，看不同的label样本在二维空间的分布。

**注意**：由于特征空间维度极高，降维难免会有损失，所以二维的可视化仅供参考。

目前支持的:

| **降维方法** | **Method名称**                                                 |
| ------------ | ------------------------------------------------------------ |
| LLE      | Standard LLE, Modified LLE                                   |
| PCA      | t-SNE, NCA                                                      |
| SVD      | Truncated SVD                                              |
| Model Based      | Random projection, Isomap, MDS, Random Trees,Spectral       |

In [ ]:
from onekey_algo.custom.components.comp1 import analysis_features
analysis_features(data[sel_feature[:-3]], data[labels[0]], methods='t-SNE')

## 构建数据

将样本的训练数据X与监督信息y分离出来，并且对训练数据进行划分，一般的划分原则为80%-20%

In [ ]:
import numpy as np
import onekey_algo.custom.components as okcomp

n_classes = 2
train_data = sel_data[(sel_data[group_info] == 'train')]
train_ids = ids[train_data.index]
train_data = train_data.reset_index()
train_data = train_data.drop('index', axis=1)
y_data = train_data[labels]
X_data = train_data.drop(labels + [group_info], axis=1)

val_data = sel_data[sel_data[group_info] != 'train']
val_ids = ids[val_data.index]
val_data = val_data.reset_index()
val_data = val_data.drop('index', axis=1)
y_val_data = val_data[labels]
X_val_data = val_data.drop(labels + [group_info], axis=1)

test_data = sel_data[sel_data[group_info] != 'train']
test_ids = ids[test_data.index]
test_data = test_data.reset_index()
test_data = test_data.drop('index', axis=1)
y_test_data = test_data[labels]
X_test_data = test_data.drop(labels + [group_info], axis=1)

y_all_data = sel_data[labels]
X_all_data = sel_data.drop(labels + [group_info], axis=1)

column_names = X_data.columns
print(f"训练集样本数：{X_data.shape}, 验证集样本数：{X_val_data.shape}, 测试集样本数：{X_test_data.shape}")

### Lasso

初始化Lasso模型，alpha为惩罚系数。具体的参数文档可以参考：[文档](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Lasso.html?highlight=lasso#sklearn.linear_model.Lasso)

### 交叉验证

不同Lambda下的，特征的的权重大小。
```python
def lasso_cv_coefs(X_data, y_data, points=50, column_names: List[str] = None, **kwargs):
    """

    Args:
        X_data: 训练数据
        y_data: 监督数据
        points: 打印多少个点。默认50
        column_names: 列名，默认为None，当选择的数据很多的时候，建议不要添加此参数
        **kwargs: 其他用于打印控制的参数。

    """
 ```

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold
import numpy as np
import matplotlib.pyplot as plt

C_range = np.logspace(-3, 1, 50)

X_train_lr = X_data.values
y_train_lr = y_data[labels[0]].values.ravel()

mean_auc_scores = []
std_auc_scores = []
coef_paths = []

cv_outer = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for C in C_range:
    lr = LogisticRegression(penalty='l1', solver='saga', C=C, max_iter=10000, random_state=42)
    auc_scores = cross_val_score(lr, X_train_lr, y_train_lr, cv=cv_outer, scoring='roc_auc')
    mean_auc_scores.append(np.mean(auc_scores))
    std_auc_scores.append(np.std(auc_scores))
    lr.fit(X_train_lr, y_train_lr)
    coef_paths.append(lr.coef_.flatten())

coef_paths = np.array(coef_paths)

best_idx = np.argmax(mean_auc_scores)
best_C = C_range[best_idx]
best_auc = mean_auc_scores[best_idx]

print(f"最佳 C 值: {best_C:.4f}")
print(f"最佳平均交叉验证 AUC: {best_auc:.4f}")

plt.figure(figsize=(10, 6))
plt.errorbar(C_range, mean_auc_scores, yerr=std_auc_scores, fmt='o-', capsize=3)
plt.axvline(best_C, color='r', linestyle='--', label=f'Best C = {best_C:.4f}')
plt.xscale('log')
plt.xlabel('C (regularization strength inverse)')
plt.ylabel('Mean CV AUC')
plt.title('Cross-Validation AUC vs. C')
plt.legend()
plt.grid(True)
plt.savefig(f'img/{task_type}Rad_feature_lr_auc.svg', bbox_inches='tight')
plt.show()

plt.figure(figsize=(12, 8))
for i in range(coef_paths.shape[1]):
    plt.plot(C_range, coef_paths[:, i], linewidth=0.8)
plt.xscale('log')
plt.xlabel('C')
plt.ylabel('Coefficient value')
plt.title('Coefficient paths for L1-regularized Logistic Regression')
plt.axvline(best_C, color='r', linestyle='--', label=f'Best C = {best_C:.4f}')
plt.legend(loc='best', fontsize='small')
plt.savefig(f'img/{task_type}Rad_feature_lr_path.svg', bbox_inches='tight')
plt.show()

final_lr = LogisticRegression(penalty='l1', solver='saga', C=best_C, max_iter=10000, random_state=42)
final_lr.fit(X_train_lr, y_train_lr)

coef = final_lr.coef_.flatten()
selected_features_lr = [col for col, c in zip(column_names, coef) if abs(c) > 0]

intercept = final_lr.intercept_[0]
formula_terms = [f"{c:+.6f} * {name}" for name, c in zip(column_names, coef) if abs(c) > 0]
formula = f"{labels[0]} = {intercept:.6f} " + " ".join(formula_terms)
print("逻辑回归模型公式：")
print(formula)

feat_coef = [(name, c) for name, c in zip(column_names, coef) if abs(c) > 0]
feat_coef_sorted = sorted(feat_coef, key=lambda x: abs(x[1]), reverse=True)
if feat_coef_sorted:
    feat_names, coef_vals = zip(*feat_coef_sorted)
    plt.figure(figsize=(10, max(6, len(feat_names)*0.3)))
    plt.barh(feat_names, coef_vals)
    plt.xlabel('Coefficient')
    plt.title('Feature coefficients (L1 logistic regression)')
    plt.tight_layout()
    plt.savefig(f'img/{labels[0]}_{task_type}Rad_feature_weights_lr.svg', bbox_inches='tight')
    plt.show()

print(f"L1 逻辑回归筛选出的特征个数: {len(selected_features_lr)}")
selected_features = [selected_features_lr]
X_data = X_data[selected_features_lr]
if 'X_val_data' in dir():
    X_val_data = X_val_data[selected_features_lr]
X_test_data = X_test_data[selected_features_lr]
print("筛选后的特征列名：")
print(X_data.columns.tolist())

## 模型筛选

根据筛选出来的数据，做模型的初步选择。当前主要使用到的是Onekey中的

1. SVM，支持向量机，引用参考。
2. KNN，K紧邻，引用参考。
3. Decision Tree，决策树，引用参考。
4. Random Forests, 随机森林，引用参考。
5. XGBoost, bosting方法。引用参考。
6. LightGBM, bosting方法，引用参考。

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, make_scorer
import joblib
import shap

models_params = {
    'LR': {
        'model': LogisticRegression(max_iter=1000, random_state=42),
        'params': {
            'classifier__C': [0.01, 0.1, 1, 10],
            'classifier__penalty': ['l2']
        }
    },
    'SVM': {
        'model': SVC(probability=True, random_state=42),
        'params': {
            'classifier__C': [0.1, 1, 10],
            'classifier__gamma': ['scale', 'auto'],
            'classifier__kernel': ['rbf']
        }
    },
    'RF': {
        'model': RandomForestClassifier(random_state=42),
        'params': {
            'classifier__n_estimators': [50, 100],
            'classifier__max_depth': [3, 5, None],
            'classifier__min_samples_split': [2, 5]
        }
    },
    'LightGBM': {
        'model': LGBMClassifier(random_state=42, verbose=-1),
        'params': {
            'classifier__n_estimators': [50, 100],
            'classifier__max_depth': [3, 5],
            'classifier__learning_rate': [0.01, 0.1]
        }
    },
    'XGBoost': {
        'model': XGBClassifier(eval_metric='logloss', use_label_encoder=False, random_state=42),
        'params': {
            'classifier__n_estimators': [50, 100],
            'classifier__max_depth': [3, 5],
            'classifier__learning_rate': [0.01, 0.1]
        }
    }
}

X_train = sel_data[sel_data[group_info] == 'train'].drop(labels + [group_info], axis=1)
y_train = sel_data[sel_data[group_info] == 'train'][labels[0]]
X_test = sel_data[sel_data[group_info] != 'train'].drop(labels + [group_info], axis=1)
y_test = sel_data[sel_data[group_info] != 'train'][labels[0]]
ids_test = ids[sel_data[sel_data[group_info] != 'train'].index]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

best_estimators = {}
cv_results = []

for name, mp in models_params.items():
    print(f"Training {name}...")
    pipeline = ImbPipeline([
        ('smote', SMOTE(random_state=42)),
        ('classifier', mp['model'])
    ])
    
    gs = GridSearchCV(
        pipeline, 
        mp['params'], 
        cv=cv, 
        scoring=make_scorer(roc_auc_score, needs_proba=True),
        n_jobs=-1,
        verbose=1
    )
    gs.fit(X_train, y_train)
    
    best_estimators[name] = gs.best_estimator_
    joblib.dump(gs.best_estimator_, f'models/{task_type}_{name}_best.pkl')
    
    cv_results.append({
        'model': name,
        'best_params': gs.best_params_,
        'best_score': gs.best_score_
    })
    print(f"{name} best CV AUC: {gs.best_score_:.4f}")

cv_df = pd.DataFrame(cv_results)
print("\nCross-validation results:")
print(cv_df)

best_model_name = cv_df.loc[cv_df['best_score'].idxmax(), 'model']
best_model = best_estimators[best_model_name]
print(f"\nBest model: {best_model_name} with CV AUC = {cv_df['best_score'].max():.4f}")

y_test_pred_proba = best_model.predict_proba(X_test)[:, 1]
test_auc = roc_auc_score(y_test, y_test_pred_proba)
print(f"Test AUC: {test_auc:.4f}")

from onekey_algo.custom.components.delong import calc_95_CI
ci = calc_95_CI(y_test, y_test_pred_proba)
print(f"95% CI: {ci[0]:.4f} - {ci[1]:.4f}")

## 保存模型结果

可以把模型预测的标签结果以及每个类别的概率都保存下来。

In [ ]:
import os
import numpy as np

os.makedirs('results', exist_ok=True)
sel_model = sel_model

for idx, label in enumerate(labels):
    for sm in sel_model:
        if sm in model_names:
            sel_model_idx = model_names.index(sm)
            target = targets[idx][sel_model_idx]
            # 预测训练集和测试集数据。
            train_indexes = np.reshape(np.array(train_ids), (-1, 1)).astype(str)
#             val_indexes = np.reshape(np.array(val_ids), (-1, 1)).astype(str)
            test_indexes = np.reshape(np.array(test_ids_sel), (-1, 1)).astype(str)
            y_train_pred_scores = target.predict_proba(X_train_sel)
#             y_val_pred_scores = target.predict_proba(X_val_sel)
            y_test_pred_scores = target.predict_proba(X_test_sel)
            columns = ['ID'] + [f"{label}-{i}"for i in range(y_test_pred_scores.shape[1])]
            # 保存预测的训练集和测试集结果
            result_train = pd.DataFrame(np.concatenate([train_indexes, y_train_pred_scores], axis=1), columns=columns)
            result_train.to_csv(f'results/{task_type}{sm}_train.csv', index=False)
#             result_val = pd.DataFrame(np.concatenate([val_indexes, y_val_pred_scores], axis=1), columns=columns)
#             result_val.to_csv(f'results/{task_type}Rad_{sm}_val.csv', index=False)
            result_test = pd.DataFrame(np.concatenate([test_indexes, y_test_pred_scores], axis=1), columns=columns)
            result_test.to_csv(f'results/{task_type}{sm}_test.csv', index=False)